# 门窗检测模型训练

**对应论文**：Section 2.2 — Faster-RCNN + ResNet50

**训练流程**：
```
数据加载（DoorWindowDataset）
  → Faster-RCNN + ResNet50-FPN
  → 内置 Loss（RPN loss + ROI loss）
  → AdamW + Cosine LR
  → 验证集 mAP 评估
  → 保存最优模型
```

## 0. 环境配置

In [4]:
import os, sys, gc, random, logging, time
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import box_iou
from numpy import genfromtxt
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── 路径配置 ──
CUBICASA_ROOT = r'/workspace/production_3d/'
DATA_FOLDER   = r'/workspace/cubicasa5k/'
CHECKPOINT_DIR = r'/workspace/production_3d/checkpoints_window_door'

os.chdir(CUBICASA_ROOT)
sys.path.insert(0, CUBICASA_ROOT)
from floortrans.loaders.house import House

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)
%matplotlib inline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✓ 环境配置完成')
print(f'  device : {DEVICE}')
if torch.cuda.is_available():
    print(f'  GPU    : {torch.cuda.get_device_name(0)}')
    print(f'  显存   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

✓ 环境配置完成
  device : cuda
  GPU    : NVIDIA GeForce RTX 5080
  显存   : 16.6 GB


## 1. 训练配置

In [ ]:
@dataclass
class DetTrainConfig:
    # 数据
    data_folder:  str = DATA_FOLDER
    train_file:   str = 'train.txt'
    val_file:     str = 'val.txt'
    window_id:    int = 1   # seg[1] 中窗的类别 id
    door_id:      int = 2   # seg[1] 中门的类别 id
    min_box_size: int = 8   # 过滤太小的框
    max_box_ratio: float = 20.0

    # 滑动窗口
    tile_size:    int = 512
    tile_overlap: int = 64

    # 增强
    aug_scale_range:      Tuple[float,float] = (0.5, 2.0)
    aug_rotation_degrees: List[int] = field(default_factory=lambda: [0,90,180,270])
    aug_use_flip: bool = True

    # 归一化
    norm_mean: Tuple = (0.485, 0.456, 0.406)
    norm_std:  Tuple = (0.229, 0.224, 0.225)

    # 模型
    num_classes: int = 3   # background / window / door
    score_thresh: float = 0.5
    nms_thresh:   float = 0.3

    # 训练超参数
    batch_size:    int   = 2     # Faster-RCNN 显存占用大，建议 2
    num_workers:   int   = 0
    max_epochs:    int   = 50
    learning_rate: float = 1e-4
    weight_decay:  float = 1e-4
    warmup_epochs: int   = 3

    # mAP 评估
    iou_threshold: float = 0.5   # IoU 阈值（标准 mAP@0.5）

    # 检查点
    checkpoint_dir: str = CHECKPOINT_DIR
    save_top_k:     int = 3

    def validate(self):
        assert Path(self.data_folder).exists()
        assert self.tile_overlap < self.tile_size // 2
        Path(self.checkpoint_dir).mkdir(parents=True, exist_ok=True)
        logger.info('✓ 配置校验通过')

CFG = DetTrainConfig()
CFG.validate()
DET_CLASSES = ['background', 'window', 'door']
print(f'batch_size={CFG.batch_size}  epochs={CFG.max_epochs}  lr={CFG.learning_rate}')
print(f'num_classes={CFG.num_classes}  iou_threshold={CFG.iou_threshold}')

## 2. 数据集

In [ ]:
# ── 复用门窗 pipeline 的函数 ──

def load_sample(data_folder, folder):
    folder   = folder.lstrip('/')
    img_path = os.path.join(data_folder, folder, 'F1_scaled.png')
    svg_path = os.path.join(data_folder, folder, 'model.svg')
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f'图像不存在: {img_path}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    house = House(svg_path, h, w)
    seg   = house.get_segmentation_tensor()
    return {'image': img, 'seg': seg}


def mask_to_boxes(mask, label, min_size=8, max_ratio=20.0):
    boxes, labels = [], []
    num_labels, components = cv2.connectedComponents(mask.astype(np.uint8), connectivity=8)
    for comp_id in range(1, num_labels):
        comp_mask = (components == comp_id).astype(np.uint8)
        contours, _ = cv2.findContours(comp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours: continue
        x, y, w, h = cv2.boundingRect(contours[0])
        if w < min_size or h < min_size: continue
        if max(w,h) / max(min(w,h),1) > max_ratio: continue
        boxes.append([x, y, x+w, y+h])
        labels.append(label)
    return boxes, labels


def augment_det_sample(image, boxes, labels,
                       scale_range=(0.5,2.0),
                       rotation_degrees=[0,90,180,270],
                       use_flip=True):
    h, w  = image.shape[:2]
    scale = random.uniform(*scale_range)
    new_h, new_w = int(h*scale), int(w*scale)
    image = cv2.resize(image, (new_w,new_h), interpolation=cv2.INTER_LINEAR)
    if len(boxes): boxes = (np.array(boxes)*scale).tolist()
    angle = random.choice(rotation_degrees)
    if angle != 0:
        h, w = image.shape[:2]
        M    = cv2.getRotationMatrix2D((w/2,h/2), angle, 1.0)
        cos, sin = abs(M[0,0]), abs(M[0,1])
        new_w = int(h*sin + w*cos)
        new_h = int(h*cos + w*sin)
        M[0,2] += (new_w-w)/2
        M[1,2] += (new_h-h)/2
        image = cv2.warpAffine(image, M, (new_w,new_h), flags=cv2.INTER_LINEAR,
                               borderMode=cv2.BORDER_REFLECT_101)
        if len(boxes):
            b = np.array(boxes, dtype=np.float32)
            corners = np.stack([
                np.column_stack([b[:,0],b[:,1]]),
                np.column_stack([b[:,2],b[:,1]]),
                np.column_stack([b[:,2],b[:,3]]),
                np.column_stack([b[:,0],b[:,3]]),
            ], axis=1).reshape(-1,2)
            ones    = np.ones((corners.shape[0],1))
            corners = (M @ np.hstack([corners,ones]).T).T.reshape(-1,4,2)
            boxes   = np.stack([
                corners[:,:,0].min(1).clip(0,new_w),
                corners[:,:,1].min(1).clip(0,new_h),
                corners[:,:,0].max(1).clip(0,new_w),
                corners[:,:,1].max(1).clip(0,new_h),
            ], axis=1).tolist()
    if use_flip and random.random() < 0.5:
        w = image.shape[1]
        image = cv2.flip(image, 1)
        if len(boxes):
            b = np.array(boxes)
            b[:,[0,2]] = w - b[:,[2,0]]
            boxes = b.tolist()
    return image, boxes, labels


class DoorWindowDataset(Dataset):
    """门窗检测数据集（懒加载）"""

    def __init__(self, cfg: DetTrainConfig, split: str = 'train'):
        self.cfg      = cfg
        self.is_train = (split == 'train')
        data_file     = cfg.train_file if self.is_train else cfg.val_file
        self.folders  = genfromtxt(cfg.data_folder + data_file, dtype='str')
        logger.info('%s Dataset: %d 个样本', split, len(self.folders))

    def __len__(self): return len(self.folders)

    def __getitem__(self, idx):
        folder = self.folders[idx]
        try:
            s = load_sample(self.cfg.data_folder, folder)
            seg = s['seg']
            door_b,   door_l   = mask_to_boxes((seg[1]==self.cfg.door_id).astype(np.uint8),
                                               label=2, min_size=self.cfg.min_box_size)
            window_b, window_l = mask_to_boxes((seg[1]==self.cfg.window_id).astype(np.uint8),
                                               label=1, min_size=self.cfg.min_box_size)
            all_boxes  = door_b  + window_b
            all_labels = door_l  + window_l

            # 随机取一个 tile
            img = s['image']
            h, w   = img.shape[:2]
            stride = self.cfg.tile_size - self.cfg.tile_overlap
            ys = sorted(set(list(range(0, max(h-self.cfg.tile_size+1,1), stride)) +
                           [max(h-self.cfg.tile_size,0)]))
            xs = sorted(set(list(range(0, max(w-self.cfg.tile_size+1,1), stride)) +
                           [max(w-self.cfg.tile_size,0)]))
            ty = random.choice(ys)
            tx = random.choice(xs)
            img_tile = img[ty:ty+self.cfg.tile_size, tx:tx+self.cfg.tile_size].copy()
            th, tw   = img_tile.shape[:2]
            if th < self.cfg.tile_size or tw < self.cfg.tile_size:
                img_tile = cv2.copyMakeBorder(img_tile,
                                              0, self.cfg.tile_size-th,
                                              0, self.cfg.tile_size-tw,
                                              cv2.BORDER_REFLECT_101)

            # 换算 box 到 tile 坐标系
            tile_boxes, tile_labels = [], []
            if all_boxes:
                b  = np.array(all_boxes, dtype=np.float32)
                cx = (b[:,0]+b[:,2])/2
                cy = (b[:,1]+b[:,3])/2
                in_tile = ((cx>=tx)&(cx<tx+self.cfg.tile_size)&
                           (cy>=ty)&(cy<ty+self.cfg.tile_size))
                if in_tile.any():
                    tb = b[in_tile].copy()
                    tb[:,[0,2]] = (tb[:,[0,2]]-tx).clip(0, self.cfg.tile_size)
                    tb[:,[1,3]] = (tb[:,[1,3]]-ty).clip(0, self.cfg.tile_size)
                    bw = tb[:,2]-tb[:,0]
                    bh = tb[:,3]-tb[:,1]
                    valid = (bw>=self.cfg.min_box_size)&(bh>=self.cfg.min_box_size)
                    tile_boxes  = tb[valid].tolist()
                    tile_labels = np.array(all_labels)[in_tile][valid].tolist()

            # 数据增强（训练时）
            if self.is_train and tile_boxes:
                img_tile, tile_boxes, tile_labels = augment_det_sample(
                    img_tile, tile_boxes, tile_labels,
                    self.cfg.aug_scale_range,
                    self.cfg.aug_rotation_degrees,
                    self.cfg.aug_use_flip,
                )
                img_tile = cv2.resize(img_tile,
                                      (self.cfg.tile_size, self.cfg.tile_size),
                                      interpolation=cv2.INTER_LINEAR)

            # 归一化 → tensor
            img_t = transforms.ToTensor()(img_tile)
            img_t = transforms.Normalize(self.cfg.norm_mean, self.cfg.norm_std)(img_t)

            if tile_boxes:
                boxes_t  = torch.as_tensor(tile_boxes,  dtype=torch.float32)
                labels_t = torch.as_tensor(tile_labels, dtype=torch.int64)
            else:
                boxes_t  = torch.zeros((0,4), dtype=torch.float32)
                labels_t = torch.zeros(0,     dtype=torch.int64)

            del s, seg
            gc.collect()
            return img_t, {'boxes': boxes_t, 'labels': labels_t}

        except Exception as e:
            logger.warning('样本 %s 失败: %s', folder, e)
            return self._empty()

    def _empty(self):
        s = self.cfg.tile_size
        return (torch.zeros(3,s,s),
                {'boxes': torch.zeros(0,4,dtype=torch.float32),
                 'labels': torch.zeros(0,dtype=torch.int64)})


def collate_fn(batch):
    """Faster-RCNN 专用：每张图框数不同，不能 stack"""
    return [b[0] for b in batch], [b[1] for b in batch]


train_ds = DoorWindowDataset(CFG, split='train')
val_ds   = DoorWindowDataset(CFG, split='val')
print(f'train: {len(train_ds)} 样本')
print(f'val  : {len(val_ds)} 样本')

imgs, targets = train_ds[0]
print(f'\n验证第一个样本：')
print(f'  image  : {imgs.shape}')
print(f'  boxes  : {targets["boxes"].shape}')
print(f'  labels : {targets["labels"].numpy()}')

## 3. 模型定义

**论文 Section 2.2**：Faster-RCNN + ResNet50-FPN

Faster-RCNN 内置了自己的 loss（RPN loss + ROI classification loss + box regression loss），
训练模式下直接返回 loss dict，不需要额外定义损失函数。

In [ ]:
def build_model(num_classes: int, score_thresh: float, nms_thresh: float):
    """
    Faster-RCNN + ResNet50-FPN
    类别：[background, window, door]
    """
    model = fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')
    # 替换分类头
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    # 推理参数
    model.roi_heads.score_thresh      = score_thresh
    model.roi_heads.nms_thresh        = nms_thresh
    model.roi_heads.detections_per_img = 100
    return model


model = build_model(
    CFG.num_classes, CFG.score_thresh, CFG.nms_thresh
).to(DEVICE)

params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'模型参数量：{params:.1f}M')

# 验证前向传播（训练模式）
model.train()
dummy_img    = [torch.randn(3,512,512).to(DEVICE)]
dummy_target = [{'boxes':  torch.tensor([[10.,10.,100.,100.]], dtype=torch.float32).to(DEVICE),
                 'labels': torch.tensor([1], dtype=torch.int64).to(DEVICE)}]
with torch.no_grad():
    loss_dict = model(dummy_img, dummy_target)
print(f'训练模式输出（loss dict）：')
for k, v in loss_dict.items():
    print(f'  {k}: {v.item():.4f}')

# 验证推理模式
model.eval()
with torch.no_grad():
    preds = model(dummy_img)
print(f'\n推理模式输出：boxes={preds[0]["boxes"].shape}  labels={preds[0]["labels"]}')

## 4. 训练工具

In [ ]:
class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self.val=self.avg=self.sum=self.count=0.0
    def update(self, val, n=1):
        self.val=val; self.sum+=val*n; self.count+=n; self.avg=self.sum/self.count


class CheckpointManager:
    def __init__(self, save_dir, top_k=3):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.top_k    = top_k
        self._records = []

    def save(self, model, optimizer, epoch, metrics):
        score = metrics.get('val_map', 0.0)
        path  = self.save_dir / f'epoch_{epoch:04d}_map_{score:.4f}.pth'
        torch.save({
            'epoch'              : epoch,
            'model_state_dict'   : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics'            : metrics,
        }, path)
        logger.info('保存检查点：%s  val_map=%.4f', path.name, score)
        self._records.append((score, path))
        self._records.sort(key=lambda x: x[0], reverse=True)
        while len(self._records) > self.top_k:
            _, old = self._records.pop()
            if old.exists(): old.unlink()


def compute_map(preds, targets, iou_threshold=0.5, num_classes=3):
    """
    简单版 mAP 计算（AP per class 取平均）
    """
    ap_list = []
    for cls in range(1, num_classes):   # 跳过 background
        tp_list, scores_list = [], []
        n_gt = 0
        for pred, target in zip(preds, targets):
            gt_boxes  = target['boxes'][target['labels']==cls]
            pd_boxes  = pred['boxes'][pred['labels']==cls]
            pd_scores = pred['scores'][pred['labels']==cls]
            n_gt     += len(gt_boxes)
            if len(pd_boxes) == 0: continue
            if len(gt_boxes) == 0:
                tp_list.extend([0]*len(pd_boxes))
                scores_list.extend(pd_scores.cpu().numpy().tolist())
                continue
            iou = box_iou(pd_boxes, gt_boxes)   # [M, N]
            matched = iou.max(dim=1).values >= iou_threshold
            tp_list.extend(matched.cpu().numpy().tolist())
            scores_list.extend(pd_scores.cpu().numpy().tolist())
        if n_gt == 0 or len(scores_list) == 0:
            ap_list.append(0.0)
            continue
        order   = np.argsort(scores_list)[::-1]
        tp_arr  = np.array(tp_list)[order]
        cum_tp  = np.cumsum(tp_arr)
        cum_fp  = np.cumsum(1 - tp_arr)
        prec    = cum_tp / (cum_tp + cum_fp + 1e-8)
        rec     = cum_tp / (n_gt + 1e-8)
        # 11-point interpolation
        ap = 0.0
        for t in np.linspace(0, 1, 11):
            p = prec[rec >= t].max() if (rec >= t).any() else 0.0
            ap += p / 11
        ap_list.append(ap)
    return float(np.mean(ap_list)) if ap_list else 0.0


ckpt_manager = CheckpointManager(CFG.checkpoint_dir, top_k=CFG.save_top_k)
print('✓ 训练工具初始化完成')

## 5. 优化器 & 学习率调度

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

optimizer = AdamW(
    model.parameters(),
    lr=CFG.learning_rate,
    weight_decay=CFG.weight_decay,
)
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=CFG.max_epochs - CFG.warmup_epochs,
    eta_min=CFG.learning_rate * 0.01,
)

print(f'优化器 : AdamW  lr={CFG.learning_rate}')
print(f'调度器 : CosineAnnealingLR  T_max={CFG.max_epochs-CFG.warmup_epochs}')

## 6. DataLoader

In [ ]:
train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=(DEVICE=='cuda'),
    collate_fn=collate_fn,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=(DEVICE=='cuda'),
    collate_fn=collate_fn,
)

print(f'train_loader : {len(train_loader)} batches')
print(f'val_loader   : {len(val_loader)} batches')

imgs, targets = next(iter(train_loader))
print(f'\nbatch: {len(imgs)} 张图')
for i, t in enumerate(targets):
    print(f'  target[{i}]: boxes={t["boxes"].shape}  labels={t["labels"].numpy()}')

## 7. 训练 & 验证函数

In [ ]:
def train_epoch(model, optimizer, loader, device, epoch, warmup_epochs, lr):
    model.train()
    meters = {k: AverageMeter() for k in
              ['loss','loss_classifier','loss_box_reg','loss_objectness','loss_rpn_box_reg']}
    t0 = time.time()

    # warmup
    if epoch <= warmup_epochs:
        lr_scale = epoch / warmup_epochs
        for pg in optimizer.param_groups:
            pg['lr'] = lr * lr_scale

    for images, targets in loader:
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k,v in t.items()} for t in targets]

        # 跳过没有框的 batch（Faster-RCNN 要求每张图至少有一个 gt box）
        if any(len(t['boxes'])==0 for t in targets):
            continue

        optimizer.zero_grad()
        loss_dict = model(images, targets)   # 训练模式直接返回 loss
        total_loss = sum(loss_dict.values())
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        n = len(images)
        meters['loss'].update(total_loss.item(), n)
        for k, v in loss_dict.items():
            if k in meters: meters[k].update(v.item(), n)

    return {k: m.avg for k,m in meters.items()}, time.time()-t0


@torch.no_grad()
def val_epoch(model, loader, device, num_classes, iou_threshold):
    model.eval()
    all_preds, all_targets = [], []

    for images, targets in loader:
        images = [img.to(device) for img in images]
        preds  = model(images)   # 推理模式返回预测结果
        all_preds.extend([{k: v.cpu() for k,v in p.items()} for p in preds])
        all_targets.extend(targets)

    val_map = compute_map(all_preds, all_targets, iou_threshold, num_classes)
    return {'val_map': val_map}


print('✓ 训练/验证函数定义完成')

## 8. 训练主循环

In [ ]:
# ── Resume 配置 ──
RESUME_CKPT = None   # 填入检查点路径即可 resume
                     # 例如：'./checkpoints_det/epoch_0005_map_0.6234.pth'

start_epoch = 1
best_map    = 0.0

if RESUME_CKPT and os.path.exists(RESUME_CKPT):
    ckpt = torch.load(RESUME_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_map    = ckpt['metrics'].get('val_map', 0.0)
    print(f'✓ Resume 成功，从 epoch {start_epoch} 继续，当前最优 map={best_map:.4f}')
else:
    print('从头开始训练')

train_history = {'loss':[], 'loss_classifier':[], 'loss_box_reg':[],
                 'loss_objectness':[], 'loss_rpn_box_reg':[], 'val_map':[]}

print(f'\n开始训练，共 {CFG.max_epochs} 个 epoch')
print(f"{'Epoch':>6}  {'total_loss':>10}  {'cls':>8}  {'box':>8}  {'rpn':>8}  {'val_map':>8}  {'lr':>8}  {'time':>6}")
print('-' * 80)

for epoch in range(start_epoch, CFG.max_epochs + 1):

    # ── 训练 ──
    train_metrics, elapsed = train_epoch(
        model, optimizer, train_loader, DEVICE, epoch,
        CFG.warmup_epochs, CFG.learning_rate,
    )

    # ── 验证 ──
    val_metrics = val_epoch(
        model, val_loader, DEVICE, CFG.num_classes, CFG.iou_threshold
    )

    # ── 学习率更新 ──
    if epoch > CFG.warmup_epochs:
        scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    # ── 记录 ──
    for k, v in train_metrics.items():
        if k in train_history: train_history[k].append(v)
    train_history['val_map'].append(val_metrics['val_map'])

    # ── 保存检查点 ──
    ckpt_manager.save(model, optimizer, epoch, {**train_metrics, **val_metrics})

    # ── 保存最优 ──
    if val_metrics['val_map'] > best_map:
        best_map = val_metrics['val_map']
        torch.save(model.state_dict(), f'{CFG.checkpoint_dir}/best_model.pth')

    # ── 打印 ──
    print(f"{epoch:>6}  "
          f"{train_metrics['loss']:>10.4f}  "
          f"{train_metrics['loss_classifier']:>8.4f}  "
          f"{train_metrics['loss_box_reg']:>8.4f}  "
          f"{train_metrics['loss_rpn_box_reg']:>8.4f}  "
          f"{val_metrics['val_map']:>8.4f}  "
          f"{current_lr:>8.6f}  "
          f"{elapsed:>5.1f}s")

print(f'\n训练完成！最优 val_map={best_map:.4f}')
print(f'最优模型保存在：{CFG.checkpoint_dir}/best_model.pth')

## 9. 训练曲线可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs = range(1, len(train_history['loss'])+1)

axes[0].plot(epochs, train_history['loss'], label='total_loss')
axes[0].set_title('Total Loss', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, train_history['loss_classifier'], label='cls_loss')
axes[1].plot(epochs, train_history['loss_box_reg'],    label='box_loss')
axes[1].plot(epochs, train_history['loss_rpn_box_reg'],label='rpn_loss')
axes[1].set_title('Loss Components', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True)

axes[2].plot(epochs, train_history['val_map'], color='green', label='val_mAP')
axes[2].axhline(y=best_map, color='red', linestyle='--', label=f'best={best_map:.4f}')
axes[2].set_title(f'Validation mAP@{CFG.iou_threshold}', fontsize=13)
axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig(f'{CFG.checkpoint_dir}/training_curves.png', dpi=120)
plt.show()

## 10. 加载最优模型验证推理

In [ ]:
# 加载最优模型
model.load_state_dict(torch.load(f'{CFG.checkpoint_dir}/best_model.pth', map_location=DEVICE))
model.eval()
print('✓ 最优模型已加载')

# 取一个验证样本
img_t, target = val_ds[0]
with torch.no_grad():
    pred = model([img_t.to(DEVICE)])[0]

print(f'GT   boxes : {target["boxes"].shape}  labels={target["labels"].numpy()}')
print(f'Pred boxes : {pred["boxes"].shape}  labels={pred["labels"].cpu().numpy()}')
print(f'Pred scores: {pred["scores"].cpu().numpy().round(3)}')

# 反归一化显示
mean   = torch.tensor(CFG.norm_mean).view(3,1,1)
std    = torch.tensor(CFG.norm_std).view(3,1,1)
img_show = (img_t * std + mean).permute(1,2,0).numpy().clip(0,1)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
color_map = {1:'blue', 2:'red'}

axes[0].imshow(img_show)
for b, l in zip(target['boxes'], target['labels']):
    x1,y1,x2,y2 = b.numpy()
    axes[0].add_patch(mpatches.Rectangle((x1,y1),x2-x1,y2-y1,
                      linewidth=2,edgecolor=color_map.get(l.item(),'gray'),facecolor='none'))
axes[0].set_title('Ground Truth', fontsize=12); axes[0].axis('off')

axes[1].imshow(img_show)
for b, l, s in zip(pred['boxes'].cpu(), pred['labels'].cpu(), pred['scores'].cpu()):
    x1,y1,x2,y2 = b.numpy()
    color = color_map.get(l.item(),'gray')
    axes[1].add_patch(mpatches.Rectangle((x1,y1),x2-x1,y2-y1,
                      linewidth=2,edgecolor=color,facecolor='none'))
    axes[1].text(x1,y1-3,f'{DET_CLASSES[l.item()]} {s:.2f}',
                color=color,fontsize=8)
axes[1].set_title('预测结果', fontsize=12); axes[1].axis('off')

legend = [mpatches.Patch(color='blue',label='window'),
          mpatches.Patch(color='red', label='door')]
axes[1].legend(handles=legend, fontsize=10)
plt.tight_layout(); plt.show()